In [67]:
import numpy as np
import nnfs
from nnfs.datasets import spiral_data
nnfs.init()

In [68]:
class Layer_Dense:
    def __init__(self, n_inputs, n_neurons):
        self.weights = 0.01 * np.random.randn(n_inputs, n_neurons)
        self.biases = np.zeros((1, n_neurons))
    
    def forward(self, inputs):
        self.inputs = inputs
        self.output = np.dot(inputs, self.weights) + self.biases

    def backward(self, dvalues):
        # Gradients on parameters
        self.dweights = np.dot(self.inputs.T, dvalues)
        self.dbiases = np.sum(dvalues, axis=0, keepdims=True)
        # Gradients on values
        self.dinputs = np.dot(dvalues, self.weights.T)

In [69]:
class Activation_ReLU:
    def forward(self, inputs):
        self.inputs = inputs
        self.output = np.maximum(0, inputs)

    def backward(self, dvalues):
        self.dinputs = dvalues.copy()
        self.dinputs[self.inputs <= 0] = 0

In [70]:
#Softmax Activation
class Activation_Softmax:
    # Forward pass
    def forward(self, inputs):
        # Get unnormalized probabilities
        exp_values = np.exp(inputs - np.max(inputs, axis=1, keepdims=True))

        # Normalize them for each sample
        probablities = exp_values / np.sum(exp_values, axis=1, keepdims=True)
        self.output = probablities

    # Backward pass
    def backward(self, dvalues):
        # Create uninitialized array
        self.dinputs = np.empty_like(dvalues)

        # Enumerate outputs and gradients
        for index, (single_output, single_dvalues) in enumerate(zip(self.output, dvalues)):
            single_output = single_output.reshape(-1,1)
            # Calculation of Jacobian matrix of the output 
            jacobian_matrix = np.diagflat(single_output) - np.dot(single_output, single_output.T)

            # Calculate sample-wise gradient
            # add it to the array of sample gradients
            self.dinputs[index] = np.dot(jacobian_matrix, single_dvalues)

In [71]:
class Loss:
    def calculate(self, output, y):
        # calculate sample loss
        sample_loss = self.forward(output, y)
        # calculate mean loss
        data_loss = np.mean(sample_loss)
        return data_loss

In [72]:
class Loss_CategoricalCrossentrophy(Loss):

    # Forward pass
    def forward(self, y_pred, y_true):
        # number of sample in a batch
        sample = len(y_pred)

        # Clip data to prevent 0
        # Clip both sides to not drag mean towards any value
        y_pred_clipped = np.clip(y_pred, 1e-7, 1-1e-7)

        # probablities for target values
        # only if categorical labels
        if len(y_true.shape) == 1:
            correct_confidences = y_pred_clipped[
                range(sample),
                y_true
            ]

        # Mask values (only for one hot encoded labels)
        elif len(y_true.shape) == 2:
            correct_confidences = np.sum(
                y_pred_clipped * y_true,
                axis=1
            )

        #Losses
        negative_log_likelihoods = -np.log(correct_confidences)
        return negative_log_likelihoods
    
    # Backward pass
    def backward(self, dvalues, y_true):
        # number of Samples
        samples = len(dvalues)
        # number of labels in every sample
        labels = len(dvalues[0])
        # If labels are sparse, turning them into one-hot vector
        if len(y_true.shape) == 1:
            y_true = np.eye(labels)[y_true]
        # Calculate gradient
        self.dinputs = -y_true / dvalues
        # Normalize gradients
        self.dinputs = self.dinputs / samples

In [73]:
class Activation_Softmax_Loss_CategoricalCrossentropy():
    # Creates activation and loss function objects
    def __init__(self):
        self.activation = Activation_Softmax()
        self.loss =  Loss_CategoricalCrossentrophy()
    # Forward pass
    def forward(self, inputs, y_true):
        # Output layer's activation function
        self.activation.forward(inputs)
        # Set the output
        self.output = self.activation.output
        # Calculate and return loss value
        return self.loss.calculate(self.output, y_true)
   
    # Backward pass
    def backward(self, dvalues, y_true):
        # Number of samples
        samples = len(dvalues)
        # If labels are one-hot encoded,
        # turn them into discrete values
        if len(y_true.shape) == 2:
            y_true = np.argmax(y_true, axis=1)
        # Copy so we can safely modify
        self.dinputs = dvalues.copy()
        # Calculate gradient
        self.dinputs[range(samples), y_true] -= 1
        # Normalize gradient
        self.dinputs = self.dinputs / samples

In [74]:
class Optimizer_SGD:
    # Initializing optimizer
    # learning rate
    def __init__(self, learning_rate=1):
        self.learning_rate = learning_rate

    def update_params(self, layer):
        layer.weights += -self.learning_rate * layer.dweights
        layer.biases += -self.learning_rate * layer.dbiases

In [75]:
# Dataset
X, y = spiral_data(samples=100, classes=3)

# Model
optimizer = Optimizer_SGD(learning_rate=0.5)
dense1 = Layer_Dense(2, 64)
activation1 = Activation_ReLU()
dense2 = Layer_Dense(64, 24)
loss_activation = Activation_Softmax_Loss_CategoricalCrossentropy()

for i in range(20000):
    # Forward pass
    dense1.forward(X)
    activation1.forward(dense1.output)
    dense2.forward(activation1.output)

    loss = loss_activation.forward(dense2.output, y)
    predictions = np.argmax(loss_activation.output, axis=1)
    if len(y.shape) == 2:
        y = np.argmax(y, axis=1)
    accuracy = np.mean(predictions == y)

    print('acc:', accuracy)
    print('loss:', loss)
    print('epoch:', i)

    # Backward pass
    loss_activation.backward(loss_activation.output, y)
    dense2.backward(loss_activation.dinputs)
    activation1.backward(dense2.dinputs)
    dense1.backward(activation1.dinputs)

    # Update weights and biases
    optimizer.update_params(dense1)
    optimizer.update_params(dense2)
    if accuracy >= 0.8:
        break

acc: 0.20666666666666667
loss: 3.1778967
epoch: 0
acc: 0.3333333333333333
loss: 3.0334063
epoch: 1
acc: 0.3333333333333333
loss: 2.8953562
epoch: 2
acc: 0.3333333333333333
loss: 2.764165
epoch: 3
acc: 0.3333333333333333
loss: 2.6401556
epoch: 4
acc: 0.3333333333333333
loss: 2.523528
epoch: 5
acc: 0.3333333333333333
loss: 2.4143414
epoch: 6
acc: 0.3333333333333333
loss: 2.3125064
epoch: 7
acc: 0.3333333333333333
loss: 2.217815
epoch: 8
acc: 0.3333333333333333
loss: 2.1299198
epoch: 9
acc: 0.3333333333333333
loss: 2.048378
epoch: 10
acc: 0.3333333333333333
loss: 1.9726691
epoch: 11
acc: 0.3333333333333333
loss: 1.9022257
epoch: 12
acc: 0.3333333333333333
loss: 1.8364689
epoch: 13
acc: 0.3333333333333333
loss: 1.7748543
epoch: 14
acc: 0.3333333333333333
loss: 1.7169026
epoch: 15
acc: 0.3333333333333333
loss: 1.6622286
epoch: 16
acc: 0.3333333333333333
loss: 1.6105692
epoch: 17
acc: 0.3333333333333333
loss: 1.5617963
epoch: 18
acc: 0.3333333333333333
loss: 1.515906
epoch: 19
acc: 0.3333333